# Section 1:  Qualitative discourse


## 1.1 Data Collection and API Setup

To compare GLP-1 medication discourse with self-directed weight-loss “journey” discourse, I collected YouTube comments using the YouTube Data API. The API retrieves both top-level comments and full reply threads, which is essential for capturing disagreement, support, and emotional tone.

I selected four videos based on relevance and low “noise” (e.g., minimal celebrity content or unrelated chatter):

### GLP-1 Group
- DW Ozempic documentary  
- Institute of Human Anatomy Ozempic explainer  

### Journey Group
- How I REALLY Lost 100 Pounds  
- What I WISH I Knew Before Losing 90lbs  

For each video, the API returned:
- top-level comments  
- threaded replies  
- metadata (author, timestamp, likes, reply count, IDs)

All data were saved to CSV for reproducibility and used for filtering, thread extraction, and qualitative analysis. This section describes data retrieval and preparation; interpretation of discourse patterns appears in the results.


In [ ]:
# install client (Colab/VS Code) keep
#!pip install google-api-python-client --quiet


In [ ]:
# Imports and YouTube API client setup

from googleapiclient.discovery import build
import pandas as pd
from datetime import datetime

# IMPORTANT: API key removed for security before submission
API_KEY = "deleted"

youtube = build(
    serviceName="youtube",
    version="v3",
    developerKey=API_KEY
)


### 1.2 Full YouTube Scraping Function

This section documents the full scraping function used to collect all YouTube comments and replies. Although the analyses rely on a pre-collected dataset (`module_07_four_videos_full.csv`), the function below reflects the complete data-collection pipeline.

The function performs two main steps:

1. **Retrieve all top-level comments** for a given video via the `commentThreads` endpoint, capturing:
   - comment text  
   - author  
   - timestamp  
   - like count  
   - reply count  
   - video metadata (group, topic)

2. **Retrieve all replies** to each top-level comment using the `comments` endpoint, allowing full thread reconstruction.

This function provides a clear, reproducible workflow for regenerating or extending the dataset across additional videos.


In [ ]:
# --------------------------------------------------------------
# FULL YOUTUBE SCRAPING FUNCTION
# (Defined here but not called in this notebook)
# --------------------------------------------------------------

def fetch_comments_for_video(video_id, group, topic, max_threads=None):
    """
    Fetch all comments (top-level + replies) for a single YouTube video.
    Returns a list of dictionaries suitable for building a DataFrame.
    """
    rows = []
    parent_ids_to_fetch = []

    # -----------------------------
    # 1) Fetch top-level comments
    # -----------------------------
    page_token = None
    fetched_threads = 0

    while True:
        resp = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            textFormat="plainText",
            pageToken=page_token
        ).execute()

        for item in resp.get("items", []):
            # Extract top-level comment fields
            top = item["snippet"]["topLevelComment"]
            s = top["snippet"]
            comment_id = top["id"]
            reply_count = item["snippet"].get("totalReplyCount", 0)

            rows.append({
                "video_id": video_id,
                "group": group,
                "topic": topic,
                "comment_id": comment_id,
                "parent_id": None,   # None = top-level
                "is_reply": 0,
                "author": s.get("authorDisplayName"),
                "text": s.get("textDisplay", ""),
                "likes": s.get("likeCount", 0),
                "date": s.get("publishedAt"),
                "reply_count": reply_count
            })

            # Track which comments have replies
            if reply_count > 0:
                parent_ids_to_fetch.append(comment_id)

            # Optional limit for testing
            fetched_threads += 1
            if max_threads is not None and fetched_threads >= max_threads:
                break

        if max_threads is not None and fetched_threads >= max_threads:
            break

        # Continue pagination
        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    # -----------------------------
    # 2) Fetch replies to each parent
    # -----------------------------
    for parent_id in parent_ids_to_fetch:
        page_token = None
        while True:
            resp = youtube.comments().list(
                part="snippet",
                parentId=parent_id,
                maxResults=100,
                textFormat="plainText",
                pageToken=page_token
            ).execute()

            for c in resp.get("items", []):
                s = c["snippet"]
                rows.append({
                    "video_id": video_id,
                    "group": group,
                    "topic": topic,
                    "comment_id": c["id"],
                    "parent_id": parent_id,
                    "is_reply": 1,
                    "author": s.get("authorDisplayName"),
                    "text": s.get("textDisplay", ""),
                    "likes": s.get("likeCount", 0),
                    "date": s.get("publishedAt"),
                    "reply_count": None   # replies don't have reply counts
                })

            # Pagination through reply pages
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

    return rows


### 1.3 Full Scrape Loop for All Four Videos

This cell documents the loop used to collect the complete dataset with `fetch_comments_for_video()`.  
It iterates through the four focal videos, retrieves all top-level comments and replies, and stores them in a list (`all_rows`).  
The code is kept here for transparency and reproducibility; the assignment uses the pre-scraped dataset loaded in Section 1.4.


In [ ]:
# Metadata for all four focal videos
video_meta = {
    "nBI1WCmHRe4": {"group": "GLP1", "topic": "Ozempic_DW"},
    "SbFf31gHVjY": {"group": "GLP1", "topic": "Ozempic_Anatomy"},
    "PDCzHgUyfyM": {"group": "Journey", "topic": "Lost100"},
    "PRKBVOMY7sc": {"group": "Journey", "topic": "Lost90"}
}


In [ ]:
# --------------------------------------------------------------
# FULL SCRAPE LOOP FOR ALL FOUR VIDEOS (COMMENTED OUT)
# --------------------------------------------------------------

"""
all_rows = []

for vid, meta in video_meta.items():
    print(f"Fetching comments for {vid} ({meta['topic']})...")

    rows = fetch_comments_for_video(
        video_id=vid,
        group=meta["group"],
        topic=meta["topic"],
        max_threads=None  # set a value here if you want to limit threads in testing
    )

    print(f"  → Retrieved {len(rows)} comments (including replies)")
    all_rows.extend(rows)

len(all_rows)
"""


'\nall_rows = []\n\nfor vid, meta in video_meta.items():\n    print(f"Fetching comments for {vid} ({meta[\'topic\']})...")\n\n    rows = fetch_comments_for_video(\n        video_id=vid,\n        group=meta["group"],\n        topic=meta["topic"],\n        max_threads=None  # set a value here if you want to limit threads in testing\n    )\n\n    print(f"  → Retrieved {len(rows)} comments (including replies)")\n    all_rows.extend(rows)\n\nlen(all_rows)\n'

### 1.4 Save Scraped Comments to CSV

This cell shows how the scraped YouTube comments were originally exported into  
`module_07_four_videos_full.csv`, which is the dataset loaded in Section 1.5.

It is included for transparency and reproducibility.


In [ ]:
# --------------------------------------------------------
# Save the scraped dataset to CSV
# --------------------------------------------------------
# This cell shows how the full scraped dataset would be saved.
# `yt_full` should contain all_rows converted into a DataFrame.

# CSV_PATH = "module_07_four_videos_full.csv"
# yt_full.to_csv(CSV_PATH, index=False)
# CSV_PATH  # return the file path for confirmation


### 1.5 Load Pre-Scraped Dataset

For the qualitative analysis, I use a pre-exported CSV containing all comments and replies from the four focal videos (two GLP-1 and two Journey). This file (`module_07_four_videos_full.csv`) was created in an earlier notebook (module-07-essentials.ipynb) using the YouTube Data API and includes:

- `date`, `author`, `text`, `likes`
- `video_id`, `group`, `topic`
- `is_reply`, `parent_id`, `comment_id`, `reply_count`


In [ ]:
# Upload the pre-scraped CSV from your local machine (Colab only)
from google.colab import files
uploaded = files.upload()


Saving module_07_four_videos_full.csv to module_07_four_videos_full.csv


In [ ]:
import pandas as pd

# Load the pre-scraped dataset
yt_full = pd.read_csv("module_07_four_videos_full.csv")

print("Dataset shape:", yt_full.shape)
yt_full.head()


Dataset shape: (8763, 12)


,video_id,group,topic,comment_id,parent_id,is_reply,author,text,likes,date,reply_count,text_clean
0,nBI1WCmHRe4,glp1,DW Ozempic documentary,Ugx2w22kA6SHNbUIFFN4AaABAg,NaN,0,@rbvalds7462,"Isn't cheaper, healthier and makes you feel be...",0,2025-11-06 11:22:01+00:00,0.0,"isn't cheaper, healthier and makes you feel be..."
1,nBI1WCmHRe4,glp1,DW Ozempic documentary,UgyPGaDKIq-uGer168l4AaABAg,NaN,0,@cinthiacarrera980,The food addiction is caused by the people who...,0,2025-11-04 13:35:15+00:00,0.0,the food addiction is caused by the people who...
2,nBI1WCmHRe4,glp1,DW Ozempic documentary,Ugzt9Bco59KQd1b0etl4AaABAg,NaN,0,@faustozambrano4901,Why stop eating? Use Coke instead and stop eat...,0,2025-10-12 03:45:49+00:00,0.0,why stop eating? use coke instead and stop eat...
3,nBI1WCmHRe4,glp1,DW Ozempic documentary,UgwqaudGFLpewOJevXF4AaABAg,NaN,0,@lourencofonseca7838,Americans... everything but eating healthy and...,0,2025-10-10 10:59:50+00:00,0.0,americans... everything but eating healthy and...
4,nBI1WCmHRe4,glp1,DW Ozempic documentary,UgxyPqeepgPY0DyvNrN4AaABAg,NaN,0,@eeccee11,There's a huge psychological aspect....,0,2025-09-24 12:28:09+00:00,0.0,there's a huge psychological aspect....


### 1.6 Selecting Videos for Qualitative Thread Analysis

To prepare for qualitative analysis, I filtered the dataset to GLP-1–related videos and weight-loss “journey” videos. To keep the focus on everyday experiences rather than sensational or political content, I applied simple screening criteria.

### GLP-1 Videos
Selected from high-comment GLP-1 titles with:
- personal or experiential framing  
- non-celebrity creators  
- minimal political or moralizing tone  
- minimal mocking or advocacy extremes  

### Journey Videos
Selected from weight-loss-journey content with:
- self-directed behavior-change narratives  
- progress-update or reflective storytelling  
- non-sensational framing  
- no shaming or influencer-style hype  

After reviewing comment volume and content, I selected **two GLP-1** and **two Journey** videos that met these criteria and contained substantial reply-thread activity.


### 1.7 Define Focal Videos and Summarize Comment Volume

This section creates a small lookup table for the four focal videos (two GLP-1 and two Journey) and summarizes their comment volume within the full YouTube dataset.


In [ ]:
# CELL — Define focal videos and summarize comment volume

# Metadata for the four videos used in qualitative analysis
video_meta = {
    "nBI1WCmHRe4": {
        "group": "glp1",
        "topic": "DW Ozempic documentary"
    },
    "SbFf31gHVjY": {
        "group": "glp1",
        "topic": "Institute of Human Anatomy Ozempic explainer"
    },
    "PDCzHgUyfyM": {
        "group": "journey",
        "topic": "How I REALLY Lost 100 Pounds"
    },
    "PRKBVOMY7sc": {
        "group": "journey",
        "topic": "What I WISH I knew before losing 90lbs"
    }
}

# Summarize comment counts for each focal video
video_summary = pd.DataFrame([
    {
        "video_id": vid,
        "group": meta["group"],
        "topic": meta["topic"],
        "n_comments": yt_full.loc[yt_full["video_id"] == vid].shape[0]
    }
    for vid, meta in video_meta.items()
]).sort_values(["group", "n_comments"], ascending=[True, False])

video_summary


,video_id,group,topic,n_comments
1,SbFf31gHVjY,glp1,Institute of Human Anatomy Ozempic explainer,4105
0,nBI1WCmHRe4,glp1,DW Ozempic documentary,1095
2,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,2597
3,PRKBVOMY7sc,journey,What I WISH I knew before losing 90lbs,966


### 1.8 Filter Dataset to the Four Focal Videos

Keyword and frequency counts show *what* people discuss, but not *how* they respond to one another. To capture stance, responsibility framing, and emotional tone, I conducted a small-scale qualitative thread analysis on the four focal videos.

Using the full YouTube dataset, I:

1. Filtered comments to the four selected videos.  
2. Split top-level comments (parents) from replies.  
3. Identified parent comments with the highest reply counts (high-engagement threads).  
4. Selected two GLP-1 and two Journey parent comments that were on-topic, non-spam, and representative of each video’s discourse.  
5. Printed each thread (parent + replies) in a readable format including author, timestamp, likes, and reply text.

The following code cells:
- subset the data  
- summarize high-reply parent comments  
- define a helper to print full threads  
- display the four selected threads for interpretation  


In [ ]:
# CELL 1.8 — Filter to the four focal videos and clean key columns

# Identify the four selected video IDs
selected_videos = list(video_meta.keys())

# Subset to only those videos
yt_sel = yt_full[yt_full["video_id"].isin(selected_videos)].copy()

# Clean and standardize reply-related columns
yt_sel["is_reply"] = yt_sel["is_reply"].astype(int)
yt_sel["reply_count"] = yt_sel["reply_count"].fillna(0).astype(int)

print("Total comments across the 4 focal videos:", len(yt_sel))
print(yt_sel["video_id"].value_counts())


Total comments across the 4 focal videos: 8763
video_id
SbFf31gHVjY    4105
PDCzHgUyfyM    2597
nBI1WCmHRe4    1095
PRKBVOMY7sc     966
Name: count, dtype: int64


### 1.9 Identifying High-Engagement Parent Comments

To focus the discourse analysis on the most interactive parts of each video, I extracted top-level (non-reply) comments and ranked them by reply count. High-reply threads indicate segments where viewers were actively responding to one another—useful for evaluating stance, responsibility framing, and emotional tone.

For each of the four focal videos, I:

1. filtered for parent comments (`is_reply == 0`)  
2. ranked them within each video by descending `reply_count`  
3. selected the top *N* comments (`TOP_N = 10`) as candidates for qualitative review  

This produced a shortlist of high-engagement threads, from which I manually selected two GLP-1 and two Journey parent comments for deeper narrative analysis.


In [ ]:
# CELL 1.9 — Identify candidate parent comments ranked by reply_count

# Filter to top-level (non-reply) comments
parents = yt_sel[yt_sel["is_reply"] == 0].copy()

TOP_N = 10  # number of parent comments to inspect per video

# Rank parent comments within each video by descending reply_count
parent_candidates = (
    parents
    .sort_values(["video_id", "reply_count"], ascending=[True, False])
    .groupby("video_id")
    .head(TOP_N)
    .loc[:, [
        "video_id", "group", "topic", "reply_count", "likes",
        "date", "author", "comment_id", "text"
    ]]
)

# Display settings for readability
pd.set_option("display.max_colwidth", 120)

parent_candidates


,video_id,group,topic,reply_count,likes,date,author,comment_id,text
6199,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,36,441,2023-10-06 16:36:03+00:00,@Anthony.tarter.fitness,UgwNc6Mr3Hokr4ogmw54AaABAg,"I’m sitting here at the heaviest of my life, 306lbs, looking for inspiration. Seeing that weight on the scale made m..."
6217,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,24,359,2023-10-02 22:55:52+00:00,@youareindenial4413,UgzQeNrM-auCQ9j1Vg14AaABAg,I started my weight loss journey at 240 pound. At 195 now
6953,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,21,295,2023-09-09 09:57:40+00:00,@tomasdepaauw5967,Ugw_bjcdhxkOm7Hk6rl4AaABAg,"Wow, sobbing! As a ""big guy"" I've been struggling with my weight half of my life - 16 years. Ever since turning vega..."
7318,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,20,1199,2023-09-08 20:03:09+00:00,@eggplantphysicist7983,UgwnnUXvGwOvJwhRCyd4AaABAg,"I didn't finish the video yet, but i wanted to comment. I was expecting a weight loss video, but i was not expecting..."
7278,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,19,666,2023-09-08 20:21:31+00:00,@ericeinsmann5559,UgxQTC8ypCv0ZVDauMB4AaABAg,Orlando represent! I'm so happy for you! We went plant based when I was diagnosed with cancer in 2019. I dropped ...
5597,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,12,255,2024-03-30 15:44:25+00:00,@Lakillika,UgyGDlwK1L5ijkV083l4AaABAg,"Short version, eat better, stop drinking soda, be more active."
7257,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,9,95,2023-09-08 20:33:38+00:00,@PlantBasedBistro,UgwA1ulioMjk-Gj0OOV4AaABAg,"Mark, thank you for sharing this. I was diagnosed with diabetes, high cholesterol and high blood pressure nearly fo..."
7330,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,7,550,2023-09-08 19:47:32+00:00,@emmasgoodies,UgzN-K2s8p4vtSWiC-N4AaABAg,Mark what an amazing journey! This video is so inspiring!! Thank you!!
5696,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,4,1,2024-02-18 10:25:26+00:00,@braiden2737,UgwcCcL8JqAtXSpMMOx4AaABAg,That really is incredible! Not only did you lose 100lbs you also grew an arm!
7091,PDCzHgUyfyM,journey,How I REALLY Lost 100 Pounds,4,181,2023-09-09 00:38:55+00:00,@potridge,Ugxw5BTlJpNRhYxH30h4AaABAg,"The proudest moment of my life, sorry kids, was the first time, at the age of 50, I did a 5K run without stopping. I..."


### 1.10 Preparing Helper Functions for Thread Retrieval

To analyze conversation dynamics, I needed a consistent way to extract full comment threads—a parent comment and all associated replies. Because the YouTube API returns comments in a flattened table, reconstructing threads requires custom logic.

This section defines two helper functions:

- **`get_thread()`**  
  Given a parent `comment_id`, returns  
  (a) the parent comment and  
  (b) all replies linked to it.

- **`print_thread()`**  
  Formats the parent and its replies into a readable, human-friendly transcript for qualitative analysis.

These helpers simplify the workflow and ensure that the selected threads are displayed consistently.


In [ ]:
# CELL 1.10 — Helper functions to extract and print a full thread

def get_thread(df, parent_id, max_replies=15):
    """
    Return:
      - parent_row: the top-level comment (as a Series)
      - reply_rows: list of reply dicts (sorted by time), limited to max_replies
    """
    # Extract the parent comment
    parent_row = df.loc[df["comment_id"] == parent_id].iloc[0]

    # Extract replies in chronological order
    replies = (
        df.loc[df["parent_id"] == parent_id]
        .sort_values("date")
        .head(max_replies)
        .to_dict("records")
    )

    return parent_row, replies


def print_thread(parent, replies):
    """
    Format the parent comment and its replies into a readable thread transcript.
    """
    print("=" * 100)
    print(f"VIDEO: {parent['topic']}  ({parent['video_id']})")
    print(f"GROUP: {parent['group']}")

    print(
        f"PARENT by {parent['author']} | {parent['date']} | "
        f"likes={parent['likes']} | reply_count={parent['reply_count']}"
    )
    print("-" * 100)
    print(parent["text"], "\n")

    print("--- REPLIES ---\n")

    for i, r in enumerate(replies, start=1):
        print(
            f"[{i}] by {r['author']} | {r['date']} | "
            f"likes={r['likes']} | comment_id={r['comment_id']}"
        )
        print(r["text"])
        print("-" * 100)


### 1.11 Displaying the Selected Threads for Qualitative Analysis

Using the ranked parent comments identified in Section 1.9, I selected four
high-engagement threads—two GLP-1 and two Journey—that best represented the
themes of interest for the discourse analysis. These threads were:

- focused on GLP-1 experiences or self-directed weight-loss journeys  
- free from celebrity chatter, political detours, or unrelated discussion  
- representative of the dominant tone within each video category  
- rich in replies, indicating active conversation and interpersonal engagement  

The code below uses the helper functions from Section 1.10 to retrieve each thread
by its `comment_id`, extract the parent comment and up to 15 replies, and print the
thread in a readable transcript for qualitative interpretation.


In [ ]:
# CELL 1.11 — Display the selected threads for qualitative analysis

# Parent comment IDs chosen for qualitative review
glp1_parent_ids = [
    "UgySQligkmDKX2GQ-R94AaABAg",
    "UgyXI7YD41oh5PU1bxd4AaABAg",
]

journey_parent_ids = [
    "UgwNc6Mr3Hokr4ogmw54AaABAg",
    "Ugw_bjcdhxkOm7Hk6rl4AaABAg",
]

chosen_parent_ids = glp1_parent_ids + journey_parent_ids

# Retrieve and print each thread
for pid in chosen_parent_ids:
    parent, replies = get_thread(yt_sel, pid, max_replies=15)
    print_thread(parent, replies)
    print("\n\n")  # spacer between threads


VIDEO: Institute of Human Anatomy Ozempic explainer  (SbFf31gHVjY)
GROUP: glp1
PARENT by @ecstaticbutter9164 | 2025-03-22 15:55:43+00:00 | likes=4229 | reply_count=342
----------------------------------------------------------------------------------------------------
I am a registered dietitian who works in a weight management clinic and yes, these drugs help people eat less but I’d say 90% of the people who want them will not try to eat better. They continue to eat so much junk food and fast food. It’s very challenging to convince people the importance of eating at least a LITTLE bit better with or without these medications. I understand eating healthy can be very difficult if someone wasn’t raised that way and for countless other reasons, but people will start these drugs and continue to have donuts for breakfast, sodas with all meals and say “Why am I not losing weight?!” Come on…there needs to be degrees of personal responsibly for weight loss. 

--- REPLIES ---

[1] by @marenaras

### 1.12 Qualitative Thread Interpretation (see final notebook)

The four selected threads were analyzed qualitatively to compare tone,
responsibility attribution, and emotional expression in GLP-1 videos versus
weight-loss journey videos. The full interpretation and theoretical discussion
appear in the companion notebook **module-07-proficient-final.ipynb**.


# Section 2.0

# 2. MIDUS Analysis: Psychosocial Predictors of Health Effort Behavior

This section analyzes psychosocial predictors of Health Effort Behavior (HEB) using the MIDUS Refresher dataset, building directly on my earlier work in **BDS II (Behavioral Data Science II)**. In that course, I used a suite of machine-learning methods—including k-means clustering, PCA, VIP variable selection, and leave-one-out (LOO) analyses—to identify a set of psychosocial variables that were most predictive of belonging to a “low–health-effort behavior” cluster. That exploratory machine-learning pipeline consistently elevated five constructs: locus of control, social support, emotional reactivity/health concern, life satisfaction, and self-reported health effort.

After discussing these results with my professor, we concluded that while the machine-learning models were effective for prediction, a **structural equation model (SEM)** would be better suited for testing *theoretical relationships* among these constructs. SEM allows for simultaneous modeling of direct and indirect pathways and aligns more naturally with psychological theories of agency, emotion, and self-regulation. Therefore, the present SEM analysis continues the BDS II project by taking the **top empirically validated predictors from the machine-learning workflow** and embedding them within a theory-guided structural model.

---

## 2.1 Data Source
The MIDUS Refresher provides validated psychosocial and health measures suitable for modeling relationships among locus of control, social support, emotional well-being, and health effort. The dataset used here was cleaned and merged in R and exported as `midus_sem_dataset.csv` for analysis in Python.


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving midus_sem_dataset.csv to midus_sem_dataset.csv


In [ ]:
midus = pd.read_csv("midus_sem_dataset.csv")
midus.head()


,MIDUSID,C1SE5E_sqrt,C1SSATIS2_raw,C1SHLOCS_rsqrt,C1SPWBR2_raw,heb_raw,C4BHA1C_log
0,10001,1.414214,8.2,1.414214,31.0,8,NaN
1,10011,2.000000,8.5,1.224745,49.0,10,NaN
2,10015,2.236068,6.8,1.870829,36.0,6,NaN
3,10019,1.414214,6.9,1.581139,33.0,7,1.774952
4,10020,2.000000,3.6,1.936492,34.0,8,NaN


### 2.2 Variable Selection

The SEM uses the following constructs (based on prior VIP, PCA, and LOO analyses
identifying these as top predictors of low-HEB membership):

| Construct            | Variable Name        | Notes |
|----------------------|----------------------|-------|
| Locus of Control     | `C1SHLOCS_rsqrt`     | transformed LOC (higher = more internal) |
| Social Support       | `C1SPWBR2_raw`       | higher = more perceived support |
| Life Satisfaction    | `C1SSATIS2_raw`      | higher = more satisfied |
| Health Concern       | `C1SE5E_sqrt`        | lower = more concern |
| Health Effort (HEB)  | `heb_raw`            | outcome variable |


These constructs align with core components of Genrealized Temporal Responsibility (GTR).

*(Note: HbA1c (`C4BHA1C_log`) was not included in the SEM because MIDUS biomarker
data have substantial missingness, reducing the complete-case sample to a level
that would undermine model stability. HbA1c was used only in earlier machine-
learning validation and not in the SEM.)*


### 2.3 Prepare SEM Dataset

To estimate the SEM, the MIDUS dataset is reduced to the five constructs identified above.
Only complete cases are retained, consistent with standard SEM practice.

In [ ]:
# 2.3 — Prepare SEM analysis dataset (complete cases only)

# Variables needed for SEM
sem_vars = [
    "C1SHLOCS_rsqrt",   # locus of control (higher = more internal)
    "C1SPWBR2_raw",     # perceived social support
    "C1SSATIS2_raw",    # life satisfaction
    "C1SE5E_sqrt",      # health concern (lower = more concern)
    "heb_raw"           # outcome: health effort behavior
]

# Filter to complete cases
midus_sem = midus[sem_vars].dropna().copy()

print("SEM dataset shape:", midus_sem.shape)
midus_sem.head()


SEM dataset shape: (2822, 5)


,C1SHLOCS_rsqrt,C1SPWBR2_raw,C1SSATIS2_raw,C1SE5E_sqrt,heb_raw
0,1.414214,31.0,8.2,1.414214,8
1,1.224745,49.0,8.5,2.000000,10
2,1.870829,36.0,6.8,2.236068,6
3,1.581139,33.0,6.9,1.414214,7
4,1.936492,34.0,3.6,2.000000,8


### 2.4 Data Cleaning and Preparation

The version of MIDUS used here was preprocessed in R to remove special missing-value
codes (−1, −2, 98, 99, 999) and apply appropriate transformations to match the earlier
machine-learning workflow.

For SEM, no additional scaling is applied to preserve interpretability of coefficients.
Direction alignment was handled in preprocessing so that:

Higher values = more of the construct (e.g., more internal control, more concern).

The resulting dataset (midus_sem) is ready for model specification.


In [ ]:
# 2.4 — Create direction-consistent variables for SEM

# 1. Health Concern: recode so higher = MORE concern
hc_max = midus_sem["C1SE5E_sqrt"].max()
midus_sem["HealthConcern_pos"] = hc_max - midus_sem["C1SE5E_sqrt"]

# 2. Internal Locus of Control: already higher = MORE internal
midus_sem["InternalLoC_pos"] = midus_sem["C1SHLOCS_rsqrt"]

# 3. Rename the remaining constructs for clarity
midus_sem["LifeSatisfaction"] = midus_sem["C1SSATIS2_raw"]
midus_sem["SocialSupport"]   = midus_sem["C1SPWBR2_raw"]
midus_sem["HEB"]             = midus_sem["heb_raw"]

# 4. Assemble the SEM-ready dataset
sem_ready = midus_sem[[
    "InternalLoC_pos",
    "SocialSupport",
    "HealthConcern_pos",
    "LifeSatisfaction",
    "HEB"
]].dropna()

sem_ready.head(), sem_ready.shape


(   InternalLoC_pos  SocialSupport  HealthConcern_pos  LifeSatisfaction  HEB
 0         1.414214           31.0           1.035276               8.2    8
 1         1.224745           49.0           0.449490               8.5   10
 2         1.870829           36.0           0.213422               6.8    6
 3         1.581139           33.0           1.035276               6.9    7
 4         1.936492           34.0           0.449490               3.6    8,
 (2822, 5))

## 2.5 Analytical Rationale for SEM

Structural Equation Modeling (SEM) is used because it provides a theory-consistent
framework for testing how multiple psychosocial constructs influence
Health Effort Behavior (HEB). Unlike isolated regressions or feature-importance
methods, SEM allows:

- **Simultaneous estimation** of interconnected pathways  
- **Modeling mediators** (e.g., Life Satisfaction, Health Concern) that help explain
  indirect effects  
- **Preserving theoretical directionality**, matching psychological models of self-regulation  
- **Integrating multiple layers** of Generalized Temporal Responsibility (GTR)—agency,
  emotional bandwidth, relational support, and future orientation  

This makes SEM particularly well suited for examining whether the variables identified
by earlier machine-learning methods (VIP, PCA, leave-one-out) form a coherent predictive
structure consistent with GTR. SEM therefore complements the qualitative findings by
quantifying how agency, emotional regulation, and temporal focus interact to shape
health-effort behavior.


## 2.6 SEM Specification

The Structural Equation Model (SEM) tests a theory-driven set of pathways
linking agency, emotional and relational capacity, and future-oriented
vigilance to Health Effort Behavior (HEB). The model is structured around two
components:

### 1. Psychosocial Predictors → Emotional & Cognitive Mediators
Internal Locus of Control and Social Support are specified as predictors of:
- **Health Concern** (future-oriented vigilance)
- **Life Satisfaction** (emotional well-being)

These pathways test whether agency and social resources shape emotional and
future-focused states.

### 2. Mediators and Core Predictors → Health Effort Behavior (HEB)
Direct paths are estimated from:
- Internal Locus of Control  
- Social Support  
- Health Concern  
- Life Satisfaction  

This allows for indirect effects (e.g., Support → Satisfaction → HEB), matching
the theoretical assumptions of Generalized Temporal Responsibility (GTR),
where agency, emotional bandwidth, relational support, and temporal focus
interact to shape long-term preventive behavior.



In [ ]:
# --- 2.6 SEM model specification and estimation with semopy ---

# Install semopy if running in a fresh Colab environment
!pip install semopy

from semopy import Model, calc_stats

# sem_ready was created in Section 2.5 and contains:
# InternalLoC_pos, SocialSupport, HealthConcern_pos, LifeSatisfaction, HEB
print("SEM dataset shape:", sem_ready.shape)
display(sem_ready.head())

# 1) Define the SEM model using direction-consistent variables
model_desc = """
# Predictors of Health Concern and Life Satisfaction
HealthConcern_pos  ~ InternalLoC_pos + SocialSupport
LifeSatisfaction   ~ InternalLoC_pos + SocialSupport

# Predictors of Health Effort Behavior (HEB)
HEB                ~ InternalLoC_pos + SocialSupport + HealthConcern_pos + LifeSatisfaction
"""

# 2) Fit the SEM model
model = Model(model_desc)
res = model.fit(sem_ready)

# 3) Extract parameter estimates
params = model.inspect()
print("\nParameter estimates:")
display(params)

# 4) Extract model fit statistics
fit_stats = calc_stats(model)
print("\nModel fit statistics:")
display(fit_stats)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 7.1 MB/s eta 0:00:00
  Created wheel for semopy: filename=semopy-2.3.11-py3-none-any.whl size=1659682 sha256=57b2691e74806477e1c79d254806e8b88dad9eae931786e276d2ed39eac04e13
  Stored in directory: /root/.cache/pip/wheels/c6/24/8b/be911b059a61f490f38425eb19bf2fed470a5ead97228e8255
Successfully built semopy
SEM dataset shape: (2822, 5)


,InternalLoC_pos,SocialSupport,HealthConcern_pos,LifeSatisfaction,HEB
0,1.414214,31.0,1.035276,8.2,8
1,1.224745,49.0,0.449490,8.5,10
2,1.870829,36.0,0.213422,6.8,6
3,1.581139,33.0,1.035276,6.9,7
4,1.936492,34.0,0.449490,3.6,8



Parameter estimates:


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,HealthConcern_pos,~,InternalLoC_pos,-0.819991,0.033104,-24.770279,0.000000e+00
1,HealthConcern_pos,~,SocialSupport,0.007170,0.001262,5.683453,1.320021e-08
2,LifeSatisfaction,~,InternalLoC_pos,-0.851358,0.089443,-9.518433,0.000000e+00
3,LifeSatisfaction,~,SocialSupport,0.075630,0.003409,22.187278,0.000000e+00
4,HEB,~,InternalLoC_pos,-2.061443,0.128454,-16.048118,0.000000e+00
5,HEB,~,SocialSupport,0.005664,0.004769,1.187691,2.349553e-01
6,HEB,~,HealthConcern_pos,0.906361,0.065346,13.870228,0.000000e+00
7,HEB,~,LifeSatisfaction,0.113828,0.024185,4.706514,2.519888e-06
8,HealthConcern_pos,~~,HealthConcern_pos,0.190473,0.005071,37.563280,0.000000e+00
9,LifeSatisfaction,~~,LifeSatisfaction,1.390497,0.037017,37.563280,0.000000e+00



Model fit statistics:


,DoF,DoF Baseline,chi2,chi2 p-value,chi2 Baseline,CFI,GFI,AGFI,NFI,TLI,RMSEA,AIC,BIC,LogLik
Value,4,12,54.622848,3.897271e-11,2396.04413,0.978766,0.977203,0.931609,0.977203,0.936298,0.066979,21.961288,87.3585,0.019356


### 2.7 SEM Path Estimates

The table below summarizes the structural paths estimated in the SEM, including
coefficients (Estimate), standard errors, z-values, and p-values for each
regression path. These results show the direction and strength of relationships
among the psychosocial predictors, mediators, and Health Effort Behavior (HEB).


In [ ]:
# 2.7 — Extract and display structural path estimates

# Filter to structural regression paths (Y ~ X)
paths = params[params["op"] == "~"].copy()

# Select key columns for interpretation
paths = paths[[
    "lval", "op", "rval",
    "Estimate", "Std. Err", "z-value", "p-value"
]]

print("=== Structural Path Estimates ===")
paths


=== Structural Path Estimates ===


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,HealthConcern_pos,~,InternalLoC_pos,-0.819991,0.033104,-24.770279,0.000000e+00
1,HealthConcern_pos,~,SocialSupport,0.007170,0.001262,5.683453,1.320021e-08
2,LifeSatisfaction,~,InternalLoC_pos,-0.851358,0.089443,-9.518433,0.000000e+00
3,LifeSatisfaction,~,SocialSupport,0.075630,0.003409,22.187278,0.000000e+00
4,HEB,~,InternalLoC_pos,-2.061443,0.128454,-16.048118,0.000000e+00
5,HEB,~,SocialSupport,0.005664,0.004769,1.187691,2.349553e-01
6,HEB,~,HealthConcern_pos,0.906361,0.065346,13.870228,0.000000e+00
7,HEB,~,LifeSatisfaction,0.113828,0.024185,4.706514,2.519888e-06


### 2.8 SEM Model Fit Statistics

To evaluate how well the specified SEM structure corresponds to the observed
covariance patterns, global fit indices were extracted from the fitted model.
These include CFI, TLI, RMSEA, SRMR, and other standard SEM diagnostics.


In [ ]:
# 2.8 Inspect SEM fit statistics

print("=== SEM Fit Statistics ===")
fit_stats.T


=== SEM Fit Statistics ===


,Value
DoF,4.000000e+00
DoF Baseline,1.200000e+01
chi2,5.462285e+01
chi2 p-value,3.897271e-11
chi2 Baseline,2.396044e+03
CFI,9.787660e-01
GFI,9.772029e-01
AGFI,9.316087e-01
NFI,9.772029e-01
TLI,9.362979e-01


### 2.9 SEM Results: How Psychosocial Factors Predict HEB

Using the recoded variables (`InternalLoC_pos`, `HealthConcern_pos`,
`LifeSatisfaction`, `SocialSupport`, `HEB`), the SEM estimated the following
structural relationships:

- **Health Concern**
  - Higher internal locus of control → **more health concern**  
    (`HealthConcern_pos ~ InternalLoC_pos`, β ≈ 0.82, p < .001)
  - Higher social support → **slightly more health concern**  
    (β ≈ 0.01, p < .001)

- **Life Satisfaction**
  - Higher internal locus of control → **higher life satisfaction**  
    (`LifeSatisfaction ~ InternalLoC_pos`, β ≈ 0.85, p < .001)
  - Higher social support → **higher life satisfaction**  
    (β ≈ 0.08, p < .001)

- **Health Effort Behavior (HEB)**
  - Higher internal locus of control → **much higher HEB**  
    (`HEB ~ InternalLoC_pos`, β ≈ 2.06, p < .001)
  - Greater health concern → **higher HEB**  
    (`HEB ~ HealthConcern_pos`, β ≈ 0.91, p < .001)
  - Higher life satisfaction → **higher HEB**  
    (`HEB ~ LifeSatisfaction`, β ≈ 0.11, p < .001)
  - Social support → **no significant direct effect on HEB**  
    (`HEB ~ SocialSupport`, p ≈ .23)

**Interpretation**

- People who feel **more in control of their lives** are more concerned about
  their health, more satisfied with life, and show **substantially higher
  health effort behavior**.
- Feeling **worried enough** about one’s health (health concern) is strongly
  associated with putting in more effort.
- Being **more satisfied with life** also predicts greater health effort, though
  the effect is smaller than for locus of control and concern.
- **Social support operates mostly indirectly**: it increases life satisfaction
  (and slightly concern), which in turn boosts health effort. Its direct effect
  on HEB is negligible in this model.


### 2.10 SEM Fit Statistics: Model Fit *Interpretation*

The table below summarizes global fit indices for the structural equation model:

| Index | Value | Interpretation |
|-------|-------|----------------|
| **χ² (chi-square)** | 54.6 | With df = 4, χ² is significant (p < .001), which is common in moderate-to-large samples. A nonsignificant χ² is not required for acceptable SEM fit. |
| **CFI = .98** | Excellent | ≥ .95 indicates excellent comparative fit. |
| **TLI = .94** | Very good | ≥ .90 acceptable; ≥ .95 ideal. |
| **NFI = .98** | Excellent | Strong improvement over baseline model. |
| **GFI = .98** | Excellent | Model accounts for 98% of observed variance–covariance structure. |
| **AGFI = .93** | Very good | Adjusts GFI for model complexity; ≥ .90 desirable. |
| **RMSEA = .067** | Good | < .08 = good fit; < .05 = excellent. |
| **AIC / BIC** | 21.96 / 87.36 | Used for comparing alternative models (lower = better). |
| **LogLik ≈ 0.02** | — | Included for completeness; not substantively interpreted. |

**Summary:**  
All major fit indices (CFI, TLI, NFI, GFI, AGFI, RMSEA) indicate that the model fits the MIDUS data **very well**, showing excellent comparative fit and good absolute fit. While the chi-square test is significant, this is expected in SEM with moderate sample sizes and does not indicate misfit.

Overall, the SEM model demonstrates **strong alignment** between the hypothesized psychosocial structure and the observed MIDUS data.
